In [1]:
import os
import dotenv
from autogen_agentchat.conditions import TextMentionTermination, TokenUsageTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from dotenv import load_dotenv
from pydantic import BaseModel

# load env vars
load_dotenv()

True

In [2]:
generation_prompt = """
You are a software developer who is helping a new software developer leave better code review comments. You are focused on providing specific feedback relating to civility. Comment on every single one of the following aspects.
"""

civility_generation_prompt = """
1. Explicit Attacks: Direct insults, name-calling, or unprofessional hostility.
2. Tonal Behaviors: Indicators of arrogance, impatience, entitlement, or frustration.
3. Nuanced Incivility: Subtle mockery, provocation, or disrespectful references to identity (gender, race, etc.).
4. Competence Questioning: Derogatory remarks regarding basic skills or labeling work as "trainwrecks."
"""

min_civility_generation_prompt = """
1. Explicit Attacks: Direct insults, name-calling, or unprofessional hostility.
"""

reflection_stop_word = "<good>"

reflection_prompt = f"""
Check each that the message has every aspect in this list covered.

{civility_generation_prompt}

Double check that the recommendation evaluated across each of the numbered aspects. Output <good> only if all elements are covered. If it needs editing. Respond with the items that were missed.
"""




In [3]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient


model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key= os.getenv("OPENAI_API_KEY")
)


dimension_name = "Civility"

generation_unit = AssistantAgent(name=f"{dimension_name}_generation_unit", model_client=model_client, model_client_stream=True, system_message=generation_prompt + civility_generation_prompt)

reflection_unit = AssistantAgent(name=f"{dimension_name}_reflection_unit", model_client=model_client, model_client_stream=True, system_message=reflection_prompt)

text_termination = TextMentionTermination(reflection_stop_word)
too_many = TokenUsageTermination(max_total_token=500)
max_msg_termination = MaxMessageTermination(max_messages=7)

team = RoundRobinGroupChat([generation_unit, reflection_unit], termination_condition=text_termination | too_many | max_msg_termination)

crc_example = "Why are you continuing to use this depreciated method when we have discussed the alternatives. You need to come ask me questions if you are confused again, this is a waste of my time and company money."

# Run the agent and stream the messages to the console.
async def main() -> None:
    await Console(team.run_stream(task=crc_example))
    # Close the connection to the model client.
    await model_client.close()



await main()


---------- TextMessage (user) ----------
Why are you continuing to use this depreciated method when we have discussed the alternatives. You need to come ask me questions if you are confused again, this is a waste of my time and company money.
---------- ModelClientStreamingChunkEvent (Civility_generation_unit) ----------
Let's break down the aspects of this comment and provide feedback on each:

1. **Explicit Attacks**: While there is no direct name-calling or personal insults, the phrase "this is a waste of my time and company money" can come across as hostile and unprofessional. It's important to maintain a respectful tone even when providing critical feedback.

2. **Tonal Behaviors**: The comment shows signs of frustration and impatience, especially with the phrases "you need to" and emphasizing that their actions are a waste of time and money. It's crucial to remain patient and understanding when guiding others, as it fosters a more positive and productive environment.

3. **Nuance